# Phase 11 — Reporting exécutif et commentaires automatisés

**Lu après :** `documentation/10_business_intelligence.ipynb` et `reports/excel/EXPLICATIONS.md`

## Pourquoi cette phase est importante

Les phases précédentes publient des fichiers, des pages HTML, un modèle Power BI et une annexe Excel. Cette phase en tire ce qu'un comité lit : un rapport court, une présentation, et des commentaires écrits par le programme.

Le risque propre à cette phase est le chiffre recopié à la main, ou le commentaire qui cite un nombre que personne ne retrouve. La règle est donc la suivante : aucun nombre n'est saisi dans un texte. Chaque nombre est lu dans un fichier du pipeline, et un contrôle le relit après l'écriture.

## Les livrables

| Livrable | Fichier | Public |
|---|---|---|
| Rapport exécutif, deux à trois pages | `reports/executive/rapport_executif.pdf` | Direction, gérant |
| Présentation, dix à quinze diapositives | `reports/management/presentation_management.pptx` | Management non technique |
| Commentaires automatisés | `reports/executive/commentaires_automatises.md` et `.json` | Relecteur avant diffusion |
| Trace du contrôle | `reports/executive/controle_des_nombres.csv` | Contrôle interne |
| Figures | `reports/executive/figures/` | Reprises dans le rapport et la présentation |

Le rapport suit cet ordre : faits marquants, recommandations avec le graphique du lien au fonds, performance contre le fonds iShares en CHF, risque, tests de résistance, qualité des données, contrôle des calculs, alertes, points à instruire. Les données, le périmètre et les limites sont écrits dans le premier paragraphe.

La présentation suit cet ordre : contexte et méthode, mandat et limites, performance, lien avec le fonds, risque, corrélation et concentration, tests de résistance, qualité des données, contrôle des calculs, alertes, recommandations, points à instruire, limites.

## Comment une phrase est écrite

Une phrase est écrite avec des **faits**. Un fait nomme un fichier du pipeline, un filtre de ligne, une colonne et une opération : lire une cellule, compter des lignes, prendre un maximum. Le programme lit la valeur, la met en forme, et garde la liste des faits de la phrase.

Les plafonds et les paramètres des chocs sont lus dans `src/config.py`, qui est le fichier où le pipeline les prend lui aussi. Les repères de reporting y sont écrits à part : la cible d'écart de suivi de la politique d'investissement, le repère de perte temporaire du profil, et le seuil de vigilance du rapport. Ce seuil n'est pas une limite du mandat, et le texte le dit.

Une alerte n'est jamais écrite d'avance. Elle apparaît quand la valeur lue franchit le repère lu. Si les données changent, la phrase change ou disparaît.

## Le contrôle des nombres

`src/reporting/number_check.py` fait le contrôle. Il relit le fichier source de chaque nombre imprimé.

1. Il extrait tous les nombres du texte : nombres décimaux à virgule, pourcentages, mois, notation scientifique. Un identifiant comme `DS-003` n'est pas un nombre du texte.
2. Pour chaque nombre, il cherche un fait déclaré dont la valeur, arrondie au nombre de décimales imprimées, donne ce nombre. Un pourcentage n'est rapproché que d'une valeur dont l'unité est déclarée.
3. Un nombre sans fait lève une erreur. Le pipeline s'arrête et la cause est dans `logs/technical.log`.

Le contrôle passe deux fois. Une première fois phrase par phrase, contre les seuls faits de la phrase. Une seconde fois sur le texte relu dans le fichier fini : le Markdown, le PDF et le PowerPoint. Cette seconde passe attrape un nombre qui aurait été tapé dans la mise en page.

Le contrôle vérifie aussi que la mention « Généré automatiquement, à relire avant diffusion. » est présente, que le rapport ne dépasse pas trois pages, et que la présentation a dix à quinze diapositives.

Ce contrôle a une limite : il prouve qu'un nombre vient d'un fichier, pas que la phrase autour du nombre est juste. C'est la raison de la relecture avant diffusion.

## Les recommandations

Cinq recommandations sont écrites par des règles, dans `src/reporting/commentary.py`. Le livre proposé est celui dont le ratio de Sharpe est le plus élevé. Les deux autres sont comparés à lui. Le lien avec le fonds est qualifié d'après la part de variance associée (`MARKET_LINK_MAJORITY`). L'avance sur le fonds n'est dite établie que si la statistique t atteint `SIGNIFICANCE_T`. Le plafond de secteur est traité d'après la capacité de la liste sous le plafond écrit. Quand les chiffres changent, le livre proposé et les phrases changent avec eux.

Ces recommandations portent sur le mandat fictif CLIENT001. Elles ne constituent ni un conseil en placement ni des ordres d'achat ou de vente : la décision revient au gérant et au client. La section « Points à instruire » liste les décisions qui restent ouvertes.

Les trois portefeuilles sont des portefeuilles hypothétiques construits rétrospectivement sur des données publiques. Il n'y a pas de relevé de dépositaire.

Les commentaires sont des phrases à trous. `src/reporting/commentary.py` les remplit à partir des fichiers, selon des règles écrites. Le rapport porte la mention « généré automatiquement, à relire avant diffusion ».

## Le contrôle des calculs repris dans le rapport

Le rapport reprend la feuille Réconciliation de l'annexe Excel : nombre de lignes OK, en EXCEPTION, non publiées, plus grand écart, tolérance. La colonne Power BI n'est remplie que par `python -m src.reporting.power_bi_export`, qui interroge le moteur local de Power BI Desktop en DAX pendant que le rapport est ouvert. Une ligne sans mesure dans le modèle reste « en attente d'export Power BI », avec sa cause.

Le rapport écrit ce que ce contrôle ne prouve pas : il ne prouve ni la détention des titres, ni les cours d'un dépositaire, ni des frais. Il ne valide pas non plus les rendements mensuels, qui sont l'entrée commune des trois lectures.

## Commande et démonstration

`python -m src.pipeline.refresh` produit les livrables à la fin de la chaîne, après l'annexe Excel. `python -m src.reporting.build_executive` ne refait que cette phase.

`tests/reporting/test_number_check.py` démontre le contrôle : les commentaires propres passent ; un nombre faux injecté dans une phrase, dans le fichier écrit, ou dans l'étape appelée par le pipeline, lève l'erreur ; un nombre ajouté sans source aussi.

La cellule suivante relit la trace du dernier contrôle.

In [1]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
trace = pd.read_csv(ROOT / 'reports' / 'executive' / 'controle_des_nombres.csv')
trace.groupby('livrable')['nombre'].count().rename('nombres relus').to_frame()

,nombres relus
livrable,
cas_d_investigation.md,29
commentaires (phrase par phrase),177
commentaires_automatises.md,177
presentation_management.pptx,197
rapport_client.pdf,159
rapport_executif.pdf,231
rapport_gerant.pdf,260


## Dictionnaire

**Fait.** Référence à une valeur d'un fichier du pipeline : fichier, ligne, colonne, opération.

**Mention.** La phrase « Généré automatiquement, à relire avant diffusion. », obligatoire sur chaque livrable.

**Seuil de vigilance.** Part d'un plafond à partir de laquelle le rapport signale un poids. C'est un réglage de lecture, pas une limite du mandat.

**Illustratif.** Simulé. Vaut pour le client, pour les trois livres et pour les quatre chocs.

**Données de conservation.** Positions, transactions, frais et flux tenus par un dépositaire. Le projet n'en a pas.